# OpenSource Clipping on Colab

**Before you run anything:**
1. Runtime > Change runtime type > **T4 GPU**
2. Click the key icon (Secrets) in the left sidebar and add `GOOGLE_API_KEY` (get one free at https://aistudio.google.com/apikey). Turn on *Notebook access*.
3. Optional secrets: `PEXELS_API_KEY` (B-roll), `HF_TOKEN` (podcast split-screen / camera-switch)

Then run the cells top to bottom.

**Optional:** for web B-roll (`BROLL_SOURCE = "web"` or `"both"`), upload a `trusted_sources.json` next to `main.py` listing channels you own or have permission to reuse. See `trusted_sources.example.json`.

In [ ]:
# Cell 1: Clone our in-house repo (public) and install dependencies
!rm -rf /content/opensource-clipping
!git clone https://github.com/sahdev15ankit-pixel/EH-Clipper.git /content/opensource-clipping
%cd /content/opensource-clipping
!pip install -q -r requirements.txt
!ffmpeg -version | head -n 1

In [ ]:
# Cell 1b: Colab compatibility fixes (numpy<2 pin vs preinstalled jax/tensorflow, PyAV version, CUDA 12 libs for Whisper)
!pip uninstall -y -q tensorflow jax jaxlib
!pip install -q "av>=14,<16" nvidia-cublas-cu12 nvidia-cudnn-cu12 deno "yt-dlp[default]"
import os, site, glob
libs = glob.glob(site.getsitepackages()[0] + "/nvidia/*/lib")
os.environ["LD_LIBRARY_PATH"] = ":".join(libs) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
print(len(libs), "CUDA lib dirs added")

In [ ]:
# Cell 2: Write API keys from Colab Secrets into .env
from pathlib import Path
from google.colab import userdata

def get_secret(name, required=False):
    try:
        return userdata.get(name)
    except Exception:
        if required:
            raise RuntimeError(f"Add {name} to Colab Secrets (key icon, left sidebar) and enable Notebook access.")
        return None

lines = [f"GOOGLE_API_KEY={get_secret('GOOGLE_API_KEY', required=True)}"]
for optional in ("PEXELS_API_KEY", "HF_TOKEN"):
    value = get_secret(optional)
    if value:
        lines.append(f"{optional}={value}")

Path(".env").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Saved .env with:", [l.split('=')[0] for l in lines])

In [ ]:
# Cell 2b: Mount Google Drive so results are saved even if Colab resets (approve the pop-up once)
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs("/content/drive/MyDrive/EH-Clipper/outputs", exist_ok=True)
print("Drive ready: MyDrive/EH-Clipper/outputs")

In [ ]:
# Cell 3: Settings and run
URL_YOUTUBE = "PASTE_YOUTUBE_URL_HERE"   # a YouTube link, OR a path to your own footage, e.g. "/content/walkthrough.mp4"
TOTAL_CLIPS = 5
RATIO = "9:16"            # 9:16, 16:9, 1:1, 3:4, 4:5
FONT_STYLE = "HINDI"      # HINDI (Devanagari + English), DEFAULT, STORYTELLER, HORMOZI, CINEMATIC
GEMINI_MODEL = "gemini-3-flash-preview"
WHISPER_COMPUTE_TYPE = "float16"   # use float32 if you hit errors
VIDEO_CQ = 19   # lower = sharper/bigger (default repo value: 23)
COOKIES = ""   # e.g. "/content/cookies.txt" only if YouTube blocks Colab ("Sign in to confirm you're not a bot")
WHISPER_LANGUAGE = ""   # "" = auto-detect, "hi" = Hindi, "en" = English

# --- Editing options (clipping/polish) ---
TIGHTEN = True          # really cut silences + um/uh/hmm before clipping (word-aligned)
GRADE = "subtle"        # none | subtle | punch | cinematic
LOUDNORM = True         # normalise clip loudness to about -16 LUFS
BROLL_SOURCE = "both"   # off | pexels | web | both   (web = channels in trusted_sources.json, see README)
REVIEW_PAGE = True      # write outputs/review.html with playable clips + copy-paste captions

extra = []
if TIGHTEN: extra.append("--tighten")
if GRADE != "none": extra.append(f"--grade {GRADE}")
if LOUDNORM: extra.append("--loudnorm")
extra.append("--no-broll" if BROLL_SOURCE == "off" else f"--broll-source {BROLL_SOURCE}")
if REVIEW_PAGE: extra.append("--review-page")
EXTRA_FLAGS = " ".join(extra)

!python main.py \
  --url "{URL_YOUTUBE}" \
  --clips {TOTAL_CLIPS} \
  --ratio "{RATIO}" \
  --font-style "{FONT_STYLE}" \
  --hook-duration 3 \
  --words-per-sub 5 \
  --gemini-model "{GEMINI_MODEL}" \
  --whisper-compute-type "{WHISPER_COMPUTE_TYPE}" \
  {("--whisper-language " + WHISPER_LANGUAGE) if WHISPER_LANGUAGE else ""} \
  {("--yt-cookies " + COOKIES) if COOKIES else ""} \
  --video-cq {VIDEO_CQ} \
  --no-bgm \
  {EXTRA_FLAGS}

# Save everything to Google Drive (runs even if the line above failed part-way)
!mkdir -p /content/drive/MyDrive/EH-Clipper && cp -r outputs/. /content/drive/MyDrive/EH-Clipper/outputs/ && echo "Saved to Drive: MyDrive/EH-Clipper/outputs"

In [ ]:
# Cell 4: Also download a zip to this computer (optional - clips are already in Drive)
!zip -rq /content/clips.zip outputs
from google.colab import files
files.download("/content/clips.zip")